In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost', 'pyarrow', 'polars'])
import os, gc
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import xgboost as xgb
import polars as pl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
# KHAI BÁO ĐƯỜNG DẪN DỮ LIỆU ĐẦU VÀO
PROCESSED_DATA_DIR = '/kaggle/input/datasets/b22dckh072/file05' 

META_PATH  = f'{PROCESSED_DATA_DIR}/filtered_metadata.parquet'
TRAIN_PATH = f'{PROCESSED_DATA_DIR}/train_interactions.parquet'
CAND_PATH  = f'{PROCESSED_DATA_DIR}/candidates_phase2.parquet'
TEST_PATH  = f'{PROCESSED_DATA_DIR}/test_interactions.parquet'
MODEL_PATH = '/kaggle/input/datasets/b22dckh072/file05/xgboost_ranking_model.json' 
FEATURES = [
    'user_total_actions', 
    'item_total_sales', 
    'price', 
    'average_rating',
    'rating_number',
    'user_avg_rating_given',
    'sasrec_score', 
    'lightgcn_score'
]
print("Đang nạp mô hình XGBoost từ đĩa cứng...")
model = xgb.Booster()
model.load_model(MODEL_PATH)
print("Nạp mô hình thành công! Đã sẵn sàng để xếp hạng.")

model.set_param({'device': 'cpu'}) 

In [ ]:
try:
    libc = ctypes.CDLL("libc.so.6")
    def trim_memory():
        libc.malloc_trim(0)
except:
    def trim_memory():
        pass

print("Đang tạo bảng đặc trưng nền bằng Polars...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)

# Trích xuất đặc trưng User giống hệt File 05
df_use = (
    lf_train.group_by('mapped_user_id').agg([
        pl.len().alias('user_total_actions').cast(pl.Float32),
        pl.col('rating').mean().alias('user_avg_rating_given').cast(pl.Float32)
    ])
    .collect()
)

# Trích xuất đặc trưng Item giống hệt File 05
df_ite = (
    lf_train.group_by('mapped_item_id').agg([
        pl.len().alias('item_total_sales').cast(pl.Float32),
        pl.col('rating').mean().alias('item_actual_avg_rating').cast(pl.Float32)
    ])
    .collect()
)

# Trích xuất giá và Metadata (nếu có dùng trong FEATURES)
df_meta_feats = (
    lf_meta.select(['mapped_item_id', 'price', 'average_rating', 'rating_number'])
    .with_columns(
        pl.col('price').cast(pl.Utf8).str.replace_all(r'[^0-9.]', '').cast(pl.Float32, strict=False).fill_null(0.0)
    )
    .unique(subset=['mapped_item_id'])
    .collect()
)

del lf_train, lf_meta
gc.collect()
trim_memory()

print("Bắt đầu chấm điểm (Chiến thuật Continuous Streaming Flush)...")

pf = pq.ParquetFile(CAND_PATH)
print(f"Tổng số dòng sẽ xử lý: {pf.metadata.num_rows:,}")

reader = pf.iter_batches(batch_size=500000)
buffer_df = pl.DataFrame()
writer = None
FINAL_TOP100_PATH = '/kaggle/working/top100_final_recommendations.parquet'

for batch in tqdm(reader, desc="Scoring Chunks"):
    chunk = pl.from_arrow(batch)
    
    # Nối toàn bộ đặc trưng vào tập Ứng viên
    chunk = chunk.join(df_use, on='mapped_user_id', how='left')
    chunk = chunk.join(df_ite, on='mapped_item_id', how='left')
    chunk = chunk.join(df_meta_feats, on='mapped_item_id', how='left')
    
    # Xử lý Score và điền giá trị Null chuẩn xác theo thiết lập huấn luyện
    chunk = chunk.with_columns([
        (1.0 / (pl.col('sasrec_rank') + 1.0)).fill_null(0.0).alias('sasrec_score'),
        (1.0 / (pl.col('lightgcn_rank') + 1.0)).fill_null(0.0).alias('lightgcn_score'),
        
        pl.col('price').fill_null(0.0),
        pl.col('average_rating').fill_null(0.0),
        pl.col('rating_number').fill_null(0.0),
        pl.col('user_total_actions').fill_null(0.0),
        pl.col('item_total_sales').fill_null(0.0),
        pl.col('user_avg_rating_given').fill_null(3.0),
        pl.col('item_actual_avg_rating').fill_null(3.0)
    ])
    
    # Chấm điểm an toàn với DMatrix
    X_cands = chunk.select(FEATURES).to_numpy()
    dtest = xgb.DMatrix(X_cands, missing=np.nan, feature_names=FEATURES)
    scores = model.predict(dtest)
    
    # Tạo bảng cực nhẹ để lưu kết quả
    chunk_res = pl.DataFrame({
        'mapped_user_id': chunk['mapped_user_id'],
        'mapped_item_id': chunk['mapped_item_id'],
        'score': pl.Series(scores, dtype=pl.Float32)
    })
    
    # Nạp vào ống đệm
    if buffer_df.height > 0:
        buffer_df = pl.concat([buffer_df, chunk_res])
    else:
        buffer_df = chunk_res
        
    del chunk, X_cands, dtest, scores, chunk_res
    gc.collect()
    
    # Lấy User cuối cùng (có thể đang bị cắt dở)
    last_user = buffer_df.get_column('mapped_user_id')[-1]
    
    # Phân tách: Những User đã hoàn tất và User cuối cùng
    completed_users_df = buffer_df.filter(pl.col('mapped_user_id') != last_user)
    buffer_df = buffer_df.filter(pl.col('mapped_user_id') == last_user)
    
    # Ghi thẳng xuống đĩa cứng
    if completed_users_df.height > 0:
        top100_completed = (
            completed_users_df
            .sort(['mapped_user_id', 'score'], descending=[False, True])
            .group_by('mapped_user_id')
            .head(100)
        )
        
        table = top100_completed.to_arrow()
        if writer is None:
            writer = pq.ParquetWriter(FINAL_TOP100_PATH, table.schema)
        writer.write_table(table)
        
        del completed_users_df, top100_completed, table
        gc.collect()
        
    trim_memory()

# Xử lý phần cặn còn lại
if buffer_df.height > 0:
    top100_last = (
        buffer_df
        .sort(['mapped_user_id', 'score'], descending=[False, True])
        .group_by('mapped_user_id')
        .head(100)
    )
    table = top100_last.to_arrow()
    if writer is None:
        writer = pq.ParquetWriter(FINAL_TOP100_PATH, table.schema)
    writer.write_table(table)

if writer is not None:
    writer.close()

print(f"ĐÃ LƯU KẾT QUẢ TẠI: {FINAL_TOP100_PATH}")